In [1]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [3]:
from langchain_core.documents import Document

In [4]:
import os
import torch

# Print available devices
print("CUDA Available:", torch.cuda.is_available())
print("PyTorch Version:", torch.__version__)

# Target the RTX 5060 Ti directly
device = torch.device("cuda:0")
print(f"Targeting: {torch.cuda.get_device_name(device)}")

# Test tensor creation and matrix multiplication on RTX 5060 Ti VRAM
x = torch.randn(5000, 5000, device=device)
y = torch.randn(5000, 5000, device=device)
z = torch.matmul(x, y)

print("\nSUCCESS: 5000x5000 matrix multiplication completed on RTX 5060 Ti!")

CUDA Available: True
PyTorch Version: 2.14.0.dev20260804+cu130
Targeting: NVIDIA GeForce RTX 5060 Ti

SUCCESS: 5000x5000 matrix multiplication completed on RTX 5060 Ti!


In [5]:
# Text data
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/python.txt", encoding="utf-8")



C:\Users\anasm\AppData\Local\Temp\ipykernel_23496\2127543635.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [7]:
document = loader.load()

In [8]:
# # pdf data
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("data/research2.pdf")
# document = pdf_loader.load()

In [9]:
# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf1_loader = PyMuPDFLoader("data/pdfs/pdf_1.pdf")

## Ingestion Pipeline

In [10]:
# Data => Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [11]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

        
            all_docs.extend(doc)
            num_docs += 1

    print(f"total pdfs : {num_docs}")
    print(f"total pages: {len(all_docs)}")
    return all_docs

In [12]:
all_pdf_documents = load_all_pdfs()

total pdfs : 2
total pages: 170


In [13]:
# chunks 
# !pip install langchain_text_splitters

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [15]:
chunks = split_docs(all_pdf_documents)
len(chunks)

480

### Embedding

In [16]:
from sentence_transformers import SentenceTransformer

In [17]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):

        self.model_name = model_name
        print("loading model...", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions =", self.model.get_embedding_dimension())

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [18]:
embedding_manager = EmbeddingManager()

loading model... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions = 384


In [19]:
import chromadb
import uuid

In [20]:
import os
import uuid
import chromadb

class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # THE FIX: Add metadata={"hnsw:space": "cosine"}
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embedding in RAG", 
                      "hnsw:space": "cosine"} 
        )

        print(f"Initialized the vector store with collection: {self.collection_name}")
        print(f"Docs in collection: {self.collection.count()}")

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")

        # store => ids, embeddings, documents, metadatas
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        # Fixed typo: 'embeding' -> 'embedding'
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            # Fixed typo: 'apped' -> 'append'
            ids.append(doc_id)

            metadata = dict(doc.metadata) if hasattr(doc, 'metadata') else {}
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)
            
            # Fixed typo: 'embedding_list' -> 'embeddings_list'
            # Assuming 'embedding' is a numpy array or tensor that needs tolist()
            embeddings_list.append(embedding.tolist() if hasattr(embedding, 'tolist') else embedding)

        # Fixed logic: Moved out of the for-loop so it executes once!
        # Fixed parameter name: 'embedding' -> 'embeddings'
        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )

        # Fixed typo: 'documnets_content' -> 'documents_content'
        print("Total documents added in vector store:", len(documents_content))
        print(f"Docs in collection: {self.collection.count()}")

In [21]:
vector_store= VectorStoreManager()

Initialized the vector store with collection: pdf_documents
Docs in collection: 0


In [22]:
# DATA => DOCUMENTS =? CHUNKS => EMBEDDINGS => STORE IN VECTOR STORE

texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

embeddings shape: (480, 384)
Total documents added in vector store: 480
Docs in collection: 480


### Retrieval Pipeline

In [23]:
from sklearn.metrics.pairwise import cosine_similarity

In [24]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.5):
        # 1. Generate the embedding for the query
        raw_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Safely handle both numpy arrays and standard lists
        query_embedding = raw_embedding.tolist() if hasattr(raw_embedding, 'tolist') else raw_embedding

        # 2. Perform semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k
        )

        retrieved_docs = []
        
        # 3. Process the results safely using .get()
        if results.get("documents") and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                
                # Calculate similarity score
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })

            print(f"Retrieved {len(retrieved_docs)} documents")

        else:
            print("No documents found")

        return retrieved_docs

In [25]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [26]:
rag_retriever.retrieve("what is encoder decoder?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
Retrieved 0 documents


[]

## Integrate RAG with LLMs

### OpenAI-GPT

In [31]:
API_KEY_OPENAI = "your_openai_api_key_here"

In [32]:
#!pip install langchain-openai

In [34]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    api_key=API_KEY_OPENAI,
    model="gpt-5.4",
    temperature=0.1,
    max_tokens=1024 
)

In [46]:
# generate our retrieval-augmented output
def generate_output(query, retriever, ll, top_k=3):
    result = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in result]) if result else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query}"""

    response = llm.invoke(prompt) # expecting a string as prompt
    return response.content

In [51]:
answer = generate_output("what is encoder decoder?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
Retrieved 0 documents
we found no relevant context for the given query


In [52]:
print(answer)

An **encoder-decoder** is a neural network architecture commonly used for tasks where you transform one sequence or representation into another.

### Simple idea
- **Encoder**: reads the input and converts it into a useful internal representation.
- **Decoder**: takes that representation and generates the output.

### Example
If the task is **English to French translation**:
- Encoder reads: `"How are you?"`
- Decoder generates: `"Comment ça va ?"`

### Where it is used
- **Machine translation**
- **Text summarization**
- **Speech recognition**
- **Image captioning**
- **Chatbots**

### In deep learning
Originally, encoder-decoder models were built using **RNNs/LSTMs**.  
Now they are often built with **Transformers**.

### Very short definition
An encoder-decoder model **encodes input into a compact representation and decodes it into the desired output**.


### Groq

In [53]:
API_KEY_GROQ= "your_groq_api_key_here"

In [55]:
#!pip install langchain-groq

In [73]:
from langchain_groq import ChatGroq

llm = ChatGroq(
     api_key = API_KEY_GROQ,
     model="llama-3.1-8b-instant",
     temperature=0.1,
     max_tokens=1024 
)

In [74]:
# generate our retrieval-augmented output
def generate_output(query, retriever, ll, top_k=3):
    result = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in result]) if result else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query}"""

    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a string as prompt
    return response.content

In [77]:
answer = generate_output("what is encoder decoder?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
Retrieved 0 documents
we found no relevant context for the given query


In [78]:
print(answer)

**Encoder-Decoder Architecture**

In the context of machine learning and deep learning, an encoder-decoder architecture is a type of neural network model that consists of two main components: an encoder and a decoder.

**Encoder:**

The encoder is responsible for taking in input data, such as text, images, or audio, and converting it into a compact, fixed-size representation, often referred to as a "code" or "embedding". This representation captures the essential features and patterns in the input data.

**Decoder:**

The decoder takes the encoded representation from the encoder and generates the output, which is typically a sequence of tokens, such as words or characters, or a continuous value, such as a probability distribution.

**Key Characteristics:**

1. **Sequential Processing:** Encoder-decoder models process input data sequentially, one element at a time.
2. **Bidirectional Processing:** The encoder typically processes the input data in both forward and backward directions, al

In [81]:
#pip install -U langchain-anthropic

In [82]:
API_KEY_ANTHROPIC= "your_anthropic_api_key_here"

In [89]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
     api_key = API_KEY_ANTHROPIC,
     model="claude-fable-5",
     # temperature=0.1,
     max_tokens=1024 
)

In [90]:
# generate our retrieval-augmented output
def generate_output(query, retriever, ll, top_k=3):
    result = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in result]) if result else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query}"""

    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a string as prompt
    return response.content

In [95]:
answer = generate_output("How are you doing?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
Retrieved 0 documents
we found no relevant context for the given query


In [96]:
print(answer)

[{'signature': 'CAISyAMKiAEIEBgCKkCqltUsTh19TwNUpUaNaKiisoaEbVB1a/T5+qDOuLkdn8yyNztuw0lBV2UGn1VZI1W+oUSV+WsW1aIcadJxysv1Mg5jbGF1ZGUtZmFibGUtNTgBQgh0aGlua2luZ1okMDMyY2E0M2MtZjhmMy00ZTFjLWE1MmYtOGJhOGVjNTY5MTZlEgyH4OjVv0+unh7J+38aDBcO1Ot6ioIzpO3M/SIwlNIjOWXt+kIsU6M2AOa8NoK7jCqucFtJO82eaaatE3Ck64Ec0HPdiGP5Y/HXvxsjKuwBUybfXTpYVBTnH9uZSRqUO/OPhimZQKFZLGbOiyRhVacPthply2iVVPpXkSE/Z/mB15TLoNFuiK8iIBoeMDh/NqJEK0MXdkmXlNpuVyTgHG8HwgUSu/dGrb6sjG63K2ERzFXdlpdLUBLJEFAPGiELTGcfR6Ttq5/mUbfCxQsP2zR69aBnSmNmOpoLD+7nsKog8d0DC2zo4oZ9InqJ0j2JaggOHic1cqzoKCaTo6DHu5gY/lRg8KZEWLl3sUWlb8yn2gs4f8SHR/jhQ/2deWOJWS5JHDK3ZOtj5soe63fMp4pR+KMoLInPoD5vrjkYAQ==', 'thinking': '', 'type': 'thinking'}, {'text': "I notice the provided context is empty, so there's no specific information to draw from. However, since your query is a general greeting, I can respond directly:\n\nI'm doing well, thank you for asking! How are you doing? Is there anything I can help you with today?\n\n*Note: If you intended to include context fo